# Chapter 4: Implementing B-Trees
*From: Database Internals*

## TL;DR

- B-Tree pages carry a **header** with metadata (magic numbers, cell counts, free-space offsets, sibling/rightmost pointers) that lets the engine validate a page and navigate between siblings without climbing back to the parent.
- Fixed-size pages meet variable-size values through **overflow pages** — linked extension pages that absorb any payload beyond `max_payload_size`, preserving a stable fanout.
- Inside a page, cells sit in insertion order while the **cell-offset array** stores them in key order. Binary search runs over the offset array and dereferences each offset to compare the underlying key (one level of indirection per probe).
- Splits and merges propagate upward using either **parent pointers** (e.g., WiredTiger) or a **breadcrumb stack** (e.g., PostgreSQL's `BTStack`) collected during the root-to-leaf descent.
- Optimizations worth knowing: **rebalancing / B\*-Trees** (defer splits by redistributing), **right-only appends** (PostgreSQL fastpath, SQLite quickbalance) for monotonic keys, **bulk loading** for presorted data, and **vacuum / page defragmentation** for reclaiming fragmented slotted pages.

## How This Chapter Fits

Chapters 2 and 3 laid down the abstract B-Tree algorithm and the generic slotted-page layout. Chapter 4 fills in the engineering details that separate a toy B-Tree from a production storage engine. The material splits into three groups:

1. **Organization**: page header, magic numbers, sibling links, rightmost pointers, high keys, overflow pages.
2. **Descent mechanics**: binary search with indirection, parent pointers vs. breadcrumbs during splits/merges.
3. **Optimization and maintenance**: rebalancing / B\*-Trees, right-only appends, bulk loading, compression, vacuum, and page defragmentation.

## Concept Map

```mermaid
graph TD
  Header["Page Header<br/>(magic, flags, free-space offsets)"] --> Sibling["Sibling Links"]
  Header --> Rightmost["Rightmost Pointer / High Key"]
  Header --> Overflow["Overflow Pages"]
  Header --> Slots["Slotted Page + Cell Offsets"]
  Slots --> Search["Binary Search with Indirection"]
  Search --> Descent["Root-to-Leaf Descent"]
  Descent --> Crumbs["Parent Pointers vs. Breadcrumbs"]
  Crumbs --> Splits["Splits / Merges Propagation"]
  Splits --> Rebalance["Rebalancing / B*-Trees"]
  Rebalance --> RightAppend["Right-Only Appends / Bulk Loading"]
  Slots --> Vacuum["Vacuum & Page Defragmentation"]
```

---

## 1. Page Header and Navigation Links

The **page header** is the first thing the engine reads when it pages in a block. It usually contains:

- A flag byte describing page type (leaf vs. internal, root, overflow, freelist).
- A cell count.
- The lower and upper free-space offsets that bracket unused bytes between the offset array and the cells.
- A magic number for validation.
- Optional navigation aids: sibling page IDs, a rightmost child pointer, or a node high key.

### Anatomy of a Slotted Page

```mermaid
graph TD
  subgraph Page["One Slotted Page"]
    H["Header<br/>magic, flags, count,<br/>free_lower, free_upper,<br/>rightmost / siblings"]
    S["Cell Offset Array<br/>(sorted by key order)"]
    F["Free Space"]
    C["Cells<br/>(insertion order, packed from end)"]
  end
  H --> S
  S --> F
  F --> C
```

### Magic Numbers

A magic number is a constant multi-byte value placed in the header so the engine can distinguish a valid page from random bytes. The book's example uses `50 41 47 45` (ASCII `PAGE`). On read, the four bytes are compared against the expected constant; a mismatch signals corruption or a misaligned read.

### Sibling Links vs. Parent Traversal

Some engines store forward and backward sibling page IDs directly in the header so a range scan can walk leaves without ascending. The trade-off is that splits and merges must update a neighbor's pointer too, which pulls an extra page into the critical section.

```mermaid
graph TD
  subgraph NoLinks["Without sibling links"]
    R1[Root] --> P1[Parent]
    P1 --> L1[Leaf A]
    P1 --> L2[Leaf B]
    L1 -. "must ascend to reach neighbor" .-> P1
  end
  subgraph WithLinks["With sibling links"]
    R2[Root] --> P2[Parent]
    P2 --> L3[Leaf A]
    P2 --> L4[Leaf B]
    L3 <--> L4
  end
```

### Rightmost Pointer and High Keys

A B-Tree internal node has N keys and N+1 child pointers. Two common layouts for the extra pointer:

- **Rightmost pointer stored separately** (e.g., SQLite) — the header carries a distinguished "last child" pointer, and the cell array holds only (key, pointer) pairs for the first N children.
- **High key / B[link]-Tree** (PostgreSQL) — an extra `K_{N+1}` key is appended so that all N+1 pointers pair with a key. The high key is the supremum of the subtree, which simplifies concurrency (see Blink-Trees in a later chapter).

```mermaid
graph LR
  subgraph A["a) Separate rightmost pointer"]
    KA["[K1 | P1] [K2 | P2] [K3 | P3] ... Prightmost (in header)"]
  end
  subgraph B["b) High-key / Blink layout"]
    KB["[K1 | P1] [K2 | P2] ... [KN | PN] [K_high | P_rightmost]"]
  end
```

### Comparison: What Different Engines Put in the Header

| Engine | Header highlights | Sibling links | Rightmost handling |
|--------|-------------------|---------------|--------------------|
| PostgreSQL | Page size, layout version, LSN, line pointers, high key | Forward + backward (Blink-Trees) | High key with paired pointer |
| InnoDB (MySQL) | Heap record count, level, index ID, page-type tag | Forward + backward (`FIL_PAGE_PREV/NEXT`) | Treated as last record in cell array |
| SQLite | Cell count, rightmost child pointer, first freeblock, fragmented-free total | No sibling links | Dedicated rightmost pointer in header |

### Trade-offs

| Design choice | Benefit | Cost |
|---------------|---------|------|
| Store sibling links | O(1) range scan across siblings | Extra page locks during split/merge |
| Separate rightmost pointer | Simpler N keys / N+1 pointers encoding | Special-case code paths for rightmost child |
| High key (Blink-Tree) | Uniform pairing; easier concurrency | One extra key per node |
| Magic number in header | Fast corruption / misalignment detection | A few wasted header bytes |

---

## 2. Overflow Pages for Variable-Size Records

Fanout assumes a fixed number of items per node, but real payloads have variable length. Without a spill mechanism, a single large value could push a node over the page size even when it technically isn't "full" yet. **Overflow pages** break the deadlock: each node keeps at most `max_payload_size = page_size / fanout` bytes of payload inline, and the remainder is written to a chain of extension pages.

```mermaid
graph LR
  Primary["Primary Page<br/>key + first max_payload_size bytes<br/>+ overflow_page_id"] --> OF1["Overflow Page 1<br/>next_overflow_id"]
  OF1 --> OF2["Overflow Page 2<br/>next_overflow_id"]
  OF2 --> OF3["Overflow Page 3<br/>next_overflow_id = NULL"]
```

### Why Store a Prefix Locally?

Keys usually have high cardinality in their first few bytes, so keeping a key prefix on the primary page lets most comparisons short-circuit without chasing an overflow pointer. The full key is still required for exact-match comparisons that reach the end of the prefix.

### Trade-offs

| Option | Pros | Cons |
|--------|------|------|
| Always inline (no overflow) | No pointer chasing; simple layout | Breaks fixed fanout; wastes space on small values |
| Spill beyond `max_payload_size` | Stable fanout; page never "stuck" without space | Extra I/O for oversized reads; bookkeeping for overflow freelist |
| Inline key prefix + overflow body | Fast key comparisons on the primary page | Hard to pick a good prefix length; updates may need to rewrite overflow chain |
| External blob store for large values | Keeps index dense; isolates hot/cold access patterns | Separate transactional story; extra indirection for reads |

When an insert exceeds `max_payload_size`, the engine checks if an existing overflow page has room; otherwise it allocates a new one and links it via `next_overflow_id` stored in the previous overflow page's header.

---

## 3. Binary Search with Indirection Pointers

Slotted pages keep cells in **insertion order** on disk so that writes are append-only within the page; only the **cell offset array** is maintained in key-sorted order. Binary search therefore operates on the offset array, and each probe dereferences one offset to read the actual key.

```mermaid
graph TD
  subgraph Offsets["Sorted Cell Offset Array"]
    O0["off[0]"] --> C2
    O1["off[1]"] --> C0
    O2["off[2]"] --> C3
    O3["off[3]"] --> C1
  end
  subgraph Cells["Cells in insertion order"]
    C0["Cell(key=40)"]
    C1["Cell(key=70)"]
    C2["Cell(key=10)"]
    C3["Cell(key=55)"]
  end
```

A probe at `mid = (lo + hi) // 2` reads `offset_array[mid]`, follows the offset to the physical cell, parses its key, and decides whether to continue left or right. This costs one extra pointer dereference per probe compared to a contiguous sorted array, but it avoids moving bytes on every insert.

### Pseudocode: Binary Search Returning a Signed Index

The chapter describes a convention where the search returns a signed integer:

- A non-negative value is the index of an exact match.
- A negative value encodes the insertion point; its absolute value is the index of the first element strictly greater than the searched key.

In [ ]:
# Binary search over a cell-offset array; cells themselves are not contiguous.
# offsets[i] points to a cell whose key is read via `get_key(page, offset)`.
# Returns i >= 0 on exact match, and -(insertion_index) - 1 on miss
# so that callers can distinguish match-at-0 from no-match-insert-at-0.

def binary_search_offsets(page, offsets, searched_key, get_key):
    lo, hi = 0, len(offsets) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        key_at_mid = get_key(page, offsets[mid])
        if key_at_mid == searched_key:
            return mid
        if key_at_mid < searched_key:
            lo = mid + 1
        else:
            hi = mid - 1
    # lo is now the insertion point (first index with key > searched_key)
    return -(lo) - 1

### Tiny Simulation of Cell-Offset Indirection

In [ ]:
# Each cell is appended at the end of the page; only the offset array is re-sorted.
class SlottedPage:
    def __init__(self):
        self.cells = []     # physical storage, insertion order
        self.offsets = []   # indices into self.cells, sorted by key

    def insert(self, key, value):
        self.cells.append((key, value))
        new_idx = len(self.cells) - 1
        # Find sorted insertion point and splice into offsets.
        lo, hi = 0, len(self.offsets)
        while lo < hi:
            mid = (lo + hi) // 2
            if self.cells[self.offsets[mid]][0] < key:
                lo = mid + 1
            else:
                hi = mid
        self.offsets.insert(lo, new_idx)

    def keys_in_order(self):
        return [self.cells[i][0] for i in self.offsets]

# Only the offset array changes on insert; cells are never rewritten.
# page = SlottedPage(); for k in [40, 10, 55, 70]: page.insert(k, "v")
# page.keys_in_order() -> [10, 40, 55, 70]; page.cells stays insertion-ordered.

### When the Upper-Level Search Misses

Internal-node lookups almost never exact-match; the B-Tree wants the first key strictly greater than the searched one to pick the correct child pointer. The negative-index convention hands this back in a single branch: the insertion point *is* the child index to descend into.

---

## 4. Breadcrumbs and Parent Pointers

When a leaf split propagates, the engine must locate the leaf's parent to insert the promoted separator. Two strategies exist.

### Strategy A — Parent Pointers in Each Node

The node stores its parent's page ID. No per-operation state is needed; the pointer is followed on demand. WiredTiger uses parent pointers for leaf traversal specifically to avoid the deadlocks that can arise with sibling pointers under concurrency.

### Strategy B — Breadcrumb Stack

During the top-down descent, the engine pushes (node, cell_index) pairs onto a stack. On a split, it pops the top entry to find the parent and the index where the new separator should go. PostgreSQL calls this `BTStack`.

```mermaid
graph TD
  R["Root<br/>cell 2 followed"] --> I["Internal<br/>cell 0 followed"]
  I --> L["Leaf (target)"]
  L -. "split propagates via popped stack" .-> I
  I -. "if full, pop again" .-> R
```

### Comparison

| Aspect | Parent pointers (WiredTiger) | Breadcrumb stack (PostgreSQL BTStack) |
|--------|------------------------------|----------------------------------------|
| Storage | Extra field in every node | In-memory stack per operation |
| Maintenance | Must update on every parent change (split/merge/rebalance) | No persistent state to update |
| Durability | Optional — pointer can be reconstructed from descent | Inherently ephemeral |
| Cost on descent | None | Push per level |
| Cost on structure change | Extra writes when parent changes | Pop per level during propagation |
| Concurrency | Simplifies sibling traversal without sibling locks | Needs revalidation if concurrent writers changed the path |

### Invalidation

Breadcrumbs can go stale if another transaction splits or merges the same subtree between the descent and the propagation. Implementations typically re-verify the cached cell index against the parent's current keyspace before trusting it; if it's invalid, they fall back to a fresh lookup.

---

## 5. Rebalancing and B\*-Trees

Splits and merges are expensive. **Rebalancing** postpones them by shifting cells between neighboring siblings first.

- On insert into a full node, try to move a few cells to a less-full left or right sibling before splitting.
- On delete, borrow from a sibling to stay at least half-full before merging.

### Standard Rebalancing

```mermaid
graph LR
  subgraph Before["Before"]
    Full["Left sibling<br/>85% full (overflowing)"]
    Empty["Right sibling<br/>40% full"]
  end
  subgraph After["After"]
    L2["Left sibling<br/>60% full"]
    R2["Right sibling<br/>60% full"]
  end
  Full -->|"move cells right, update parent separator"| L2
  Empty --> R2
```

Whenever cells cross a sibling boundary, the parent's separator key must be rewritten to preserve the invariant `K_{i-1} <= K_s < K_i`.

### B\*-Tree Split: 2-to-3

Knuth's **B\*-Tree** takes the idea further. It keeps redistributing until *both* siblings are full, and only then performs a **2-to-3 split**: two full siblings become three nodes each at two-thirds occupancy. SQLite uses this variant.

```mermaid
graph TD
  subgraph BStar["B*-Tree 2-to-3 split"]
    A0["Sibling A (full)"]
    B0["Sibling B (full)"]
    A0 --> A1["Node A'<br/>~2/3 full"]
    A0 --> M1["Node M<br/>~2/3 full"]
    B0 --> M1
    B0 --> B1["Node B'<br/>~2/3 full"]
  end
```

### Trade-offs

| Variant | Average occupancy | Tree height | Split frequency | Code complexity |
|---------|-------------------|-------------|-----------------|-----------------|
| Plain B-Tree (split on overflow) | ~50% right after split | Taller | Higher | Simplest |
| B-Tree with rebalance | ~65–75% | Medium | Lower | Moderate (neighbor lookup, parent update) |
| B\*-Tree (2-to-3 split) | ~66%+ steady-state | Shortest | Lowest | Highest (three-way balancing logic) |

Higher occupancy pays off on reads: fewer pages to traverse, better cache utilization, less wasted I/O per query.

---

## 6. Right-Only Appends and Bulk Loading

### Right-Only Appends

Monotonically increasing keys (auto-increment IDs, time-series timestamps) concentrate all inserts on the rightmost leaf. Two engines have named optimizations for this pattern:

- **PostgreSQL fastpath**: cache the rightmost leaf page. If the new key is strictly greater than its first key and the page has room, insert there and skip the root-to-leaf descent entirely.
- **SQLite quickbalance**: when an insert lands on the far right of a *full* rightmost node, allocate a brand-new rightmost node and just add its pointer to the parent. The new page starts nearly empty but is expected to fill up quickly, saving the cost of a rebalance.

```mermaid
graph LR
  subgraph Fastpath["PostgreSQL fastpath"]
    NewKey["Insert key > firstKey(R)"] --> R["Cached rightmost leaf"]
    R --> Done["Skip descent"]
  end
  subgraph Quickbalance["SQLite quickbalance"]
    Full["Full rightmost node"] --> Alloc["Allocate new rightmost leaf"]
    Alloc --> Parent["Append pointer to parent"]
  end
```

### Bulk Loading

When presorted data is available (restore from dump, rebuild after defrag, index build on a large existing table), the engine can bypass the normal algorithm entirely and compose the tree **bottom-up**:

1. Fill a leaf page completely from the sorted input and write it out.
2. Propagate the leaf's first key up to an in-memory internal-node buffer.
3. When an internal-node buffer fills, write it and propagate its first key to the level above.
4. Repeat until the input is exhausted, then flush the partial internal nodes.

```mermaid
graph TD
  subgraph L2["Level 2 (root)"]
    R["Root: K_A, K_B, K_C"]
  end
  subgraph L1["Level 1 (internal)"]
    I1["[first keys of L0 leaves 1..k]"]
    I2["[first keys of L0 leaves k+1..m]"]
    I3["[first keys of L0 leaves m+1..n]"]
  end
  subgraph L0["Level 0 (leaves, written left-to-right)"]
    P1["Leaf 1 (full)"]
    P2["Leaf 2 (full)"]
    P3["..."]
    Pn["Leaf n (partial)"]
  end
  R --> I1
  R --> I2
  R --> I3
  I1 --> P1
  I1 --> P2
  I2 --> P3
  I3 --> Pn
```

### Mutable vs. Immutable Bulk-Loaded Trees

| Aspect | Mutable bulk load | Immutable bulk load |
|--------|-------------------|---------------------|
| Target fill factor | ~70–80% (leave room for inserts) | 100% |
| Splits during construction | None (right-only appends) | None |
| Splits after load | Yes — needs reserved free space | Never — tree is read-only |
| Use case | Initial load / index rebuild | SSTables, snapshot indexes, LSM levels |

### Pseudocode: Bulk-Load Driver

In [ ]:
# Bottom-up bulk load from a sorted iterator. Simplified: ignores serialization,
# overflow pages, and writing to disk -- just shows the level propagation.
def bulk_load(sorted_records, leaf_fanout, internal_fanout):
    # Level 0: pack leaves greedily from the sorted stream.
    leaves = []
    buf = []
    for rec in sorted_records:
        buf.append(rec)
        if len(buf) == leaf_fanout:
            leaves.append(buf)
            buf = []
    if buf:
        leaves.append(buf)

    # Higher levels: each node holds `internal_fanout` first-keys from the level below.
    level = [(leaf[0][0], leaf) for leaf in leaves]   # (separator, child)
    while len(level) > 1:
        next_level = []
        for i in range(0, len(level), internal_fanout):
            chunk = level[i:i + internal_fanout]
            separator = chunk[0][0]
            next_level.append((separator, chunk))
        level = next_level
    # level[0] is the root (separator, children)
    return level[0]

---

## 7. Vacuum and Page Defragmentation

Slotted pages accumulate junk over time. The core causes:

- **Deletes** only clear the cell offset, leaving the cell body in place (it becomes "garbage" — unreachable from the root).
- **Updates** on leaves may write a new cell without removing the old one, especially when MVCC keeps prior versions visible to concurrent readers.
- **Splits** truncate the offset array, marking the tail cells unreachable even though their bytes remain.

The cumulative effect: a page may have enough *logical* free space but no contiguous run to fit a new cell.

```mermaid
graph LR
  subgraph Fragmented["Fragmented Slotted Page"]
    H["Header"]
    Off["Offset Array (3 live pointers)"]
    Free1["free gap"]
    G1["garbage cell"]
    Live1["live cell"]
    Free2["free gap"]
    G2["garbage cell"]
    Live2["live cell"]
    Live3["live cell"]
    H --> Off --> Free1 --> G1 --> Live1 --> Free2 --> G2 --> Live2 --> Live3
  end
```

### The Vacuum / Compaction Loop

Vacuum walks pages asynchronously and rewrites fragmented ones:

1. Read all live cells (those with offsets in the array).
2. Sort them by key and pack them contiguously at the end of the page.
3. Rebuild the offset array.
4. Reset the free-space offsets.
5. If the page is relocated, add its old page ID to the **free page list** (freelist) so it can be reused.

The freelist must itself be persisted across crashes — otherwise the database leaks pages. SQLite, for example, keeps "trunk" pages in a linked list, each holding addresses of freed pages.

### Synchronous vs. Asynchronous Rewrites

| Mode | When it runs | Pros | Cons |
|------|--------------|------|------|
| Synchronous on write | Right before an insert that doesn't fit contiguously | Avoids creating an unnecessary overflow page | Adds latency to the write path |
| Asynchronous vacuum | Background worker | Keeps user-facing latency stable | Needs throttling and coordination with MVCC readers |

### Compression Sidebar

Page-wise compression pairs naturally with fixed-size pages: each page compresses and decompresses independently on load/flush. The catch is disk-block alignment — a compressed page smaller than a disk block still costs a full block to read, so the win is mainly on hot-but-repetitive data.

```mermaid
graph LR
  subgraph Case1["a) Compressed page smaller than block"]
    B1["Disk block<br/>compressed page + padding of neighbor data"]
  end
  subgraph Case2["b) Page spans multiple blocks"]
    B2a["Block 1"] --> B2b["Block 2"]
  end
```

| Compression granularity | Pros | Cons |
|-------------------------|------|------|
| Whole file | Highest ratio | Requires full-file rewrite on any update |
| Page-wise | Independent load/flush; fits slotted-page design | Wasted bytes when compressed page < disk block |
| Row-wise | Decouples compression from page layout | Lower ratio than page-level |
| Column-wise | Great ratio for analytical workloads | Doesn't fit row-store B-Trees naturally |

### Tiny Page-Compaction Simulation

In [ ]:
# Rewrites the page so that live cells are packed contiguously and in key order.
def compact(page_cells, live_offsets):
    # page_cells: list of (key, value) tuples in insertion order (may contain garbage)
    # live_offsets: offsets (indices) of the cells still reachable from the offset array
    live = [page_cells[i] for i in live_offsets]
    live.sort(key=lambda kv: kv[0])
    # After compaction, the offset array is rebuilt sequentially.
    new_cells = live
    new_offsets = list(range(len(live)))
    return new_cells, new_offsets

---

## Cross-Concept Trade-off Matrix

| Concept | What it optimizes | What it costs |
|---------|-------------------|---------------|
| Sibling links | Fast range scans | Extra locks on split/merge |
| High keys (Blink) | Concurrency, uniform layout | One extra key per node |
| Overflow pages | Stable fanout with variable payloads | Extra I/O for oversized reads |
| Offset-array indirection | Zero cell movement on insert | One dereference per binary-search probe |
| Parent pointers | Simple upward navigation | Writes on every parent change |
| Breadcrumbs | Zero persistent overhead | Revalidation under concurrency |
| Rebalancing / B\* | Higher occupancy, shorter tree | More bookkeeping, more complex splits |
| Right-only fastpath | Skips descent for monotonic keys | Dead weight when workload is random |
| Bulk loading | No splits, 100% fill possible | Requires presorted input |
| Vacuum | Reclaims fragmented space | CPU + I/O for rewrites; freelist durability |

---

## Engine-by-Engine Cheat Sheet

| Mechanism | PostgreSQL | SQLite | InnoDB (MySQL) | WiredTiger |
|-----------|-----------|--------|----------------|-----------|
| Rightmost handling | High key (Blink-Tree) | Dedicated rightmost pointer in header | Last record in cell array | High-key style |
| Sibling links | Forward + backward | No | Forward + backward | Uses parent pointers instead |
| Parent navigation | Breadcrumbs (`BTStack`) | Breadcrumbs during balance | Per-operation cursor stack | Parent pointers |
| Rebalancing variant | Split on overflow | B\*-Tree (`balance-siblings`) | Split on overflow | Split on overflow |
| Right-only optimization | Fastpath (cached rightmost leaf) | Quickbalance | Similar to fastpath | Append cursor |
| Vacuum model | Asynchronous `VACUUM` + autovacuum | Autovacuum / incremental vacuum | Purge thread + insert buffer | Background reconciliation |

---

## Key Takeaways

1. A B-Tree page's header does double duty: it validates the page (magic number) and accelerates navigation (sibling links, rightmost pointer, high key, free-space offsets).
2. Variable-size payloads don't break fixed fanout because overflow pages absorb any bytes beyond `max_payload_size`; storing a key prefix locally keeps most comparisons off the overflow chain.
3. Inside a page, cells sit in insertion order while an offset array holds the logical key order — so binary search runs on the offset array with one dereference per probe, and inserts never shuffle cell bytes.
4. Upward navigation during splits and merges is either via parent pointers in each node (WiredTiger) or a breadcrumb stack collected on descent (PostgreSQL `BTStack`). Each trades persistent state against in-memory bookkeeping.
5. Rebalancing and B\*-Tree 2-to-3 splits raise average occupancy, shortening the tree and shrinking read cost, at the price of more intricate balancing code.
6. Right-only append optimizations (PostgreSQL fastpath, SQLite quickbalance) exploit monotonic keys by skipping the read path; bulk loading generalizes this to build a tree bottom-up from sorted input with zero splits.
7. Vacuum / page defragmentation rewrites fragmented slotted pages into logical key order, reclaims dead cells, maintains a durable free page list, and may pair with page-wise compression to trade CPU for I/O.

---

## See Also

- [[01-page-header-and-navigation-links]]
- [[02-overflow-pages]]
- [[03-binary-search-with-indirection-pointers]]
- [[04-breadcrumbs-and-parent-pointers]]
- [[05-rebalancing-and-b-star-trees]]
- [[06-right-only-appends-and-bulk-loading]]
- [[07-vacuum-and-page-defragmentation]]